In [2]:
import os
os.chdir(os.path.dirname(os.getcwd()))

In [3]:
import polars as pl
import pandas as pd
import numpy as np
def dummy_npwarn_decorator_factory():
  def npwarn_decorator(x):
    return x
  return npwarn_decorator
np._no_nep50_warning = getattr(np, '_no_nep50_warning', dummy_npwarn_decorator_factory)
from statsmodels.tsa.stattools import acf, pacf
from scipy.stats import pearsonr
from utils.metrics import crps, quantile_loss
from tqdm import tqdm

In [16]:
df_arima = pd.read_csv('logs/m5/ARIMA.csv')
df_arima = df_arima[12350:]
df_arima

,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
12350,HOBBIES_1_001_CA_1_0.005_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
12351,HOBBIES_1_002_CA_1_0.005_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
12352,HOBBIES_1_003_CA_1_0.005_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
12353,HOBBIES_1_004_CA_1_0.005_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
12354,HOBBIES_1_005_CA_1_0.005_validation,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
771115,FOODS_3_823_WI_3_0.995_evaluation,2.889064,3.047455,2.729115,2.797279,3.201407,2.968582,3.194073,2.990503,3.121097,...,3.184588,3.129474,3.281970,3.118417,3.196650,3.168817,3.141887,3.222819,3.185741,3.302427
771116,FOODS_3_824_WI_3_0.995_evaluation,2.192903,2.193882,2.196025,2.196719,2.197888,2.198012,2.198911,2.199284,2.199838,...,2.207285,2.208089,2.208953,2.209784,2.210627,2.211471,2.212325,2.213164,2.214019,2.214859
771117,FOODS_3_825_WI_3_0.995_evaluation,4.001474,3.848391,3.959204,3.998929,4.067123,4.055092,4.110247,4.083672,4.130335,...,4.206643,4.215837,4.221906,4.231159,4.238521,4.246639,4.254074,4.262182,4.269614,4.278052
771118,FOODS_3_826_WI_3_0.995_evaluation,4.052581,4.204671,4.257540,4.257097,4.263639,4.266858,4.276928,4.282611,4.291082,...,4.329642,4.334139,4.339829,4.344746,4.350145,4.352540,4.357811,4.362455,4.367322,4.371679


In [7]:
selling_prices = pd.read_csv('data/m5-forecasting-accuracy/sell_prices.csv')
selling_prices

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26
...,...,...,...,...
6841116,WI_3,FOODS_3_827,11617,1.00
6841117,WI_3,FOODS_3_827,11618,1.00
6841118,WI_3,FOODS_3_827,11619,1.00
6841119,WI_3,FOODS_3_827,11620,1.00


In [ ]:
# Extract quantile
df_arima["quantile"] = df_arima["id"].str.extract(r"_(0\.\d+)_validation$").astype(float)
# Remove '_quantile_validation' at the end
df_arima["id_no_quantile"] = df_arima["id"].str.replace(r"_0\.\d+_validation$", "", regex=True)
# Split id_no_quantile from the right
split_cols = df_arima["id_no_quantile"].str.rsplit("_", n=2, expand=True)
df_arima["item_id"] = split_cols[0]
df_arima["store_id"] = split_cols[1]

# Reshape to long format
df_long = df_arima.melt(
    id_vars=["id", "item_id", "store_id", "quantile"],
    value_vars=[f"F{i}" for i in range(1, 29)],
    var_name="day",
    value_name="forecast"
)

# Clean and format
df_long["day"] = df_long["day"].str.extract("F(\d+)").astype(int)

<>:19: SyntaxWarning: invalid escape sequence '\d'
<>:19: SyntaxWarning: invalid escape sequence '\d'
/var/folders/y0/x4h3jn3n1fb8xx5p0h2h2sqc0000gn/T/ipykernel_22979/1576139340.py:19: SyntaxWarning: invalid escape sequence '\d'
  df_long["day"] = df_long["day"].str.extract("F(\d+)").astype(int)


In [18]:
# drop ones that have missing value in iter_id or store_id
df_long = df_long.dropna(subset=["item_id", "store_id"])
df_long

,id,item_id,store_id,quantile,day,forecast
0,HOBBIES_1_001_CA_1_0.005_validation,HOBBIES_1_001,CA_1,0.005,1,0.000000
1,HOBBIES_1_002_CA_1_0.005_validation,HOBBIES_1_002,CA_1,0.005,1,0.000000
2,HOBBIES_1_003_CA_1_0.005_validation,HOBBIES_1_003,CA_1,0.005,1,0.000000
3,HOBBIES_1_004_CA_1_0.005_validation,HOBBIES_1_004,CA_1,0.005,1,0.000000
4,HOBBIES_1_005_CA_1_0.005_validation,HOBBIES_1_005,CA_1,0.005,1,0.000000
...,...,...,...,...,...,...
21245555,FOODS_3_823_WI_3_0.995_evaluation,FOODS_3_823_WI_3,0.995_evaluation,NaN,28,3.302427
21245556,FOODS_3_824_WI_3_0.995_evaluation,FOODS_3_824_WI_3,0.995_evaluation,NaN,28,2.214859
21245557,FOODS_3_825_WI_3_0.995_evaluation,FOODS_3_825_WI_3,0.995_evaluation,NaN,28,4.278052
21245558,FOODS_3_826_WI_3_0.995_evaluation,FOODS_3_826_WI_3,0.995_evaluation,NaN,28,4.371679


In [8]:
sales = pd.read_csv('data/m5-forecasting-accuracy/sales_train_evaluation.csv')
sales["id"] = sales["id"].str.replace("_validation", "", regex=False)

# Parameters
H = 1
dm = 1941

# 2. Add empty forecast columns: d_1942, ..., d_{1941+H}
for h in range(1, H + 1):
    sales[f"d_{dm + h}"] = np.nan

# 3. Melt to long format (wide → long)
id_vars = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
value_vars = [col for col in sales.columns if col.startswith("d_")]
sales_long = sales.melt(id_vars=id_vars, value_vars=value_vars, 
                        var_name="d", value_name="demand")

# 4. Extract day index from "d" (e.g. "d_1942" → 1942)
sales_long["d"] = sales_long["d"].str[2:].astype(int)
sales_long

,id,item_id,dept_id,cat_id,store_id,state_id,d,demand
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,1,0.0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,1,0.0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,1,0.0
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,1,0.0
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,1,0.0
...,...,...,...,...,...,...,...,...
59211575,FOODS_3_823_WI_3_evaluation,FOODS_3_823,FOODS_3,FOODS,WI_3,WI,1942,NaN
59211576,FOODS_3_824_WI_3_evaluation,FOODS_3_824,FOODS_3,FOODS,WI_3,WI,1942,NaN
59211577,FOODS_3_825_WI_3_evaluation,FOODS_3_825,FOODS_3,FOODS,WI_3,WI,1942,NaN
59211578,FOODS_3_826_WI_3_evaluation,FOODS_3_826,FOODS_3,FOODS,WI_3,WI,1942,NaN


In [11]:
ptmax = 730
dm = 1941  # from earlier
lag_buffer = 375

sales_long = sales_long[sales_long["d"] >= dm - ptmax - lag_buffer].copy()
sales_long

,id,item_id,dept_id,cat_id,store_id,state_id,d,demand
25459150,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,836,0.0
25459151,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,836,0.0
25459152,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,836,0.0
25459153,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,836,2.0
25459154,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,836,0.0
...,...,...,...,...,...,...,...,...
59211575,FOODS_3_823_WI_3_evaluation,FOODS_3_823,FOODS_3,FOODS,WI_3,WI,1942,NaN
59211576,FOODS_3_824_WI_3_evaluation,FOODS_3_824,FOODS_3,FOODS,WI_3,WI,1942,NaN
59211577,FOODS_3_825_WI_3_evaluation,FOODS_3_825,FOODS_3,FOODS,WI_3,WI,1942,NaN
59211578,FOODS_3_826_WI_3_evaluation,FOODS_3_826,FOODS_3,FOODS,WI_3,WI,1942,NaN


In [15]:
n_item = 3049
n_stores = 10
n_all = n_item * n_stores
n_dates = 1941

# Keep only the first 3049 unique item_ids
selected_items = sales_long["item_id"].unique()[:n_item]
sales_long = sales_long[sales_long["item_id"].isin(selected_items)].copy()
sales_long = sales_long.sort_values(by=["item_id", "store_id", "d"])
sales_long

,id,item_id,dept_id,cat_id,store_id,state_id,d,demand
25460762,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,836,1.0
25491252,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,837,1.0
25521742,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,838,3.0
25552232,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,839,0.0
25582722,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,840,1.0
...,...,...,...,...,...,...,...,...
59088182,HOUSEHOLD_2_516_WI_3_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,1938,0.0
59118672,HOUSEHOLD_2_516_WI_3_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,1939,0.0
59149162,HOUSEHOLD_2_516_WI_3_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,1940,0.0
59179652,HOUSEHOLD_2_516_WI_3_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,1941,0.0


In [18]:
print(sorted(list(map(int, sales_long['d'].unique()))))

[836, 837, 838, 839, 840, 841, 842, 843, 844, 845, 846, 847, 848, 849, 850, 851, 852, 853, 854, 855, 856, 857, 858, 859, 860, 861, 862, 863, 864, 865, 866, 867, 868, 869, 870, 871, 872, 873, 874, 875, 876, 877, 878, 879, 880, 881, 882, 883, 884, 885, 886, 887, 888, 889, 890, 891, 892, 893, 894, 895, 896, 897, 898, 899, 900, 901, 902, 903, 904, 905, 906, 907, 908, 909, 910, 911, 912, 913, 914, 915, 916, 917, 918, 919, 920, 921, 922, 923, 924, 925, 926, 927, 928, 929, 930, 931, 932, 933, 934, 935, 936, 937, 938, 939, 940, 941, 942, 943, 944, 945, 946, 947, 948, 949, 950, 951, 952, 953, 954, 955, 956, 957, 958, 959, 960, 961, 962, 963, 964, 965, 966, 967, 968, 969, 970, 971, 972, 973, 974, 975, 976, 977, 978, 979, 980, 981, 982, 983, 984, 985, 986, 987, 988, 989, 990, 991, 992, 993, 994, 995, 996, 997, 998, 999, 1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029,